# Datamine COKRIG Process Example

This notebook demonstrates advanced multivariate/univariate grade estimation and Ordinary Kriging using the **`cokrig`** process wrapper in `dmstudio`.

In [ ]:
# Step 1: Connect to Datamine and Verify Active Project
import os
import shutil
import glob
import pandas as pd
from dmstudio import dmcommands, dmfiles, initialize, special, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Prepare Inputs and COKRIG-compatible parameters
We copy the drillhole data and search parameter files locally, and construct custom variogram set, fields, and estimation parameter files using the `t_` prefix.

In [ ]:
# Copy drillholes and search parameters locally using t_ prefix
project_folder = oScript.ActiveProject.Folder
repo_root = os.path.abspath(os.path.join(project_folder, '..', '..'))
help_db = os.path.join(repo_root, 'tutorials', 'Database', 'DMTutorials', 'Data', 'VBOP', 'Datamine')

    shutil.copy(os.path.join(repo_root, 'tutorials', 'comp_holes.dmx'), os.path.join(notebook_folder, 't_comp_holes.dmx'))
shutil.copy(os.path.join(help_db, "_vb_spar.dmx"), "t_search_params.dmx")

# Generate model prototype
print("Generating prototype...")
dm_fil.protom(
    out_o='t_model_proto',
    rotmod_p=0,
    arguments=" 'N' 'Y' '5900' '4800' '-100' '10' '10' '10' '25' '45' '32'"
)
# 1. Create t_kriging_vmodel
vpar_src = agent.read_datamine(os.path.join(help_db, "_vb_vpar.dmx"))
vpar_src['GRADE'] = ['CU', 'AU', 'DENSITY']
vpar_src['GRADE2'] = ['CU', 'AU', 'DENSITY']
vpar_src['VSETNUM'] = [2.0, 1.0, 3.0]
vpar_src.to_csv('t_kriging_vmodel.csv', index=False)
special.inpfil(csv='t_kriging_vmodel.csv', out_o='t_kriging_vmodel')

# 2. Create t_kriging_epar
epar_df = pd.DataFrame({
    'EREFNUM': [1.0, 2.0, 3.0],
    'VSETNUM': [1.0, 2.0, 3.0],
    'SREFNUM': [1.0, 1.0, 2.0],
    'IMETHOD': [3.0, 3.0, 3.0],
    'DISCX': [3.0, 3.0, 3.0],
    'DISCY': [3.0, 3.0, 3.0],
    'DISCZ': [3.0, 3.0, 3.0],
    'PARENT': [1.0, 1.0, 1.0]
})
epar_df.to_csv('t_kriging_epar.csv', index=False)
special.inpfil(csv='t_kriging_epar.csv', out_o='t_kriging_epar')

# 3. Create t_kriging_fields
fields_df = pd.DataFrame({
    'EREFNUM': [1.0, 2.0, 3.0],
    'IN_VAR': ['AU', 'CU', 'DENSITY'],
    'EST': ['AU_EST', 'CU_EST', 'DENS_EST'],
    'VAR': ['AU_VAR', 'CU_VAR', 'DENS_VAR'],
    'NUMSAMP': ['AU_NS', 'CU_NS', 'DENS_NS']
})
fields_df.to_csv('t_kriging_fields.csv', index=False)
special.inpfil(csv='t_kriging_fields.csv', out_o='t_kriging_fields')

# Clean up local temporary CSVs
for f in ['t_kriging_vmodel.csv', 't_kriging_epar.csv', 't_kriging_fields.csv']:
    if os.path.exists(f):
        os.remove(f)

print("Setup complete.")

## Step 3: Run COKRIG estimation
COKRIG executes Ordinary Kriging on AU, CU, and DENSITY simultaneously using the custom parameters.

In [ ]:
# Run COKRIG
# Note: Wrap in try-except in case Advanced Estimation module is not licensed
print("Running advanced kriging estimation (COKRIG)...")
try:
    dm_cmd.cokrig(
        samples_i=os.path.join(notebook_folder, 't_comp_holes'),
        proto_i=os.path.join(notebook_folder, 't_model_proto'),
        fields_i=os.path.join(notebook_folder, 't_kriging_fields'),
        epar_i=os.path.join(notebook_folder, 't_kriging_epar'),
        spar_i=os.path.join(notebook_folder, 't_search_params'),
        vmodel_i=os.path.join(notebook_folder, 't_kriging_vmodel'),
        outmodel_o=os.path.join(notebook_folder, 't_grade_model_cokrig_result'),
        xpt_f='X',
        ypt_f='Y',
        zpt_f='Z'
    )
    print("COKRIG execution finished.")
except Exception as e:
    print("COKRIG failed with exception:", e)

## Step 4: Verify and Load Results in Pandas

In [ ]:
    print(f"COKRIG block model cells: {len(df_cokrig)}")
    print("\nCOKRIG Gold (AU_EST) Grade Summary:")
    print(df_cokrig[df_cokrig['AU_EST'] > -99]['AU_EST'].describe())
else:
    print("COKRIG output not found. Please verify your Advanced Estimation license.")

## Step 5: Clean up Project Folder
Run this cell to clean up all copied and generated files, leaving only this notebook.

In [ ]:
# Cleanup local files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    os.remove(f)
    print(f"Removed {os.path.basename(f)}")